In [12]:
import pandas as pd
import numpy as np
import pystormtracker as pst
import xarray as xr
import json 

In [57]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = (np.sin((p2 - p1) / 2) ** 2
         + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2)
    return 2 * R * np.arcsin(np.sqrt(a))

def storm_disk(ds, time, lat, lon, radius_km=1000):
    deg = radius_km / 111.0 * 1.5                     
    lon_pad = deg / max(np.cos(np.radians(lat)), 0.2) # longitude degrees shrink with latitude
    sub = ds.sel(
        time=time,
        lat=slice(lat - deg, lat + deg),         
        lon=slice(lon - lon_pad, lon + lon_pad),
    )
    LON, LAT = xr.broadcast(sub.lon, sub.lat)
    dist = haversine_km(lat, lon, LAT, LON)
    return sub.where(dist <= radius_km)

In [58]:
with open("/Users/annasophiamaxen/Desktop/PYTHON/master_thesis/PyStormTracker/json/tracks_feb_04.trackjson") as f:
    d = json.load(f)

data, v = d["data"], d["data"]["variables"]

In [59]:
df = pd.DataFrame({
    "time":     pd.to_datetime(data["times"], unit="ms"),
    "lat":      data["lats"],
    "lon":      data["lons"],
    "vo":       v["vo"],
    "major_km": v["object_moment_major_axis_km"],
    "minor_km": v["object_moment_minor_axis_km"],
})

In [60]:
df

,time,lat,lon,vo,major_km,minor_km
0,2004-02-17 00:00:00,63.75,-36.00,0.000077,1779.046565,1162.453295
1,2004-02-17 06:00:00,63.75,-33.25,0.000131,2726.649339,1244.548719
2,2004-02-17 12:00:00,64.50,-31.75,0.000121,3461.112559,865.245498
3,2004-02-17 18:00:00,67.75,-27.25,0.000104,2763.977729,1162.079154
4,2004-02-18 00:00:00,72.25,-22.50,0.000156,2762.092223,1173.620012
...,...,...,...,...,...,...
129,2004-02-20 06:00:00,80.00,-31.75,0.000021,1542.612375,474.896740
130,2004-02-20 12:00:00,80.00,-21.25,0.000085,1742.856752,881.073233
131,2004-02-20 18:00:00,80.00,-11.00,0.000091,1688.818966,654.526259
132,2004-02-21 00:00:00,76.00,6.00,0.000065,1574.356360,626.909554


In [61]:
df["track_id"] = np.repeat(d["index"]["ids"], np.diff(d["index"]["offsets"]))

In [62]:
df['track_id']

0       23
1       23
2       23
3       23
4       23
      ... 
129    673
130    673
131    673
132    673
133    673
Name: track_id, Length: 134, dtype: int64

In [67]:
ds = xr.open_dataset("/Users/annasophiamaxen/Desktop/PYTHON/master_thesis/PyStormTracker/Data/vars_feb_04_stepType-instant.nc").rename({"valid_time": "time", "latitude": "lat", "longitude": "lon"})
if float(ds.lon.max()) > 180:
    ds = ds.assign_coords(lon=((ds.lon + 180) % 360) - 180).sortby("longitude")

ds = ds.sortby("lat")  

df["on_edge"] = (
    (df.lat >= 79.99) | (df.lat <= 30.01) |
    (df.lon >= 39.99) | (df.lon <= -79.99)
)

df["radius_km"] = (df.major_km / 2).clip(lower=300, upper=1500)

lat_lo, lat_hi = float(ds.lat.min()), float(ds.lat.max())
lon_lo, lon_hi = float(ds.lon.min()), float(ds.lon.max())

inside = df.lat.between(lat_lo, lat_hi) & df.lon.between(lon_lo, lon_hi)
work = df[inside & ~df.on_edge]

In [71]:
out, skipped = [], []
for r in work.itertuples():                          # work, not df[~df.on_edge]
    disk = storm_disk(ds, r.time, r.lat, r.lon, r.radius_km)
    if disk is None or disk["msl"].count() == 0:
        skipped.append((r.track_id, r.time, r.lat, r.lon))
        continue
    ws = np.hypot(disk["u10"], disk["v10"])
    out.append({
        "track_id": r.track_id, "time": r.time, "lat": r.lat, "lon": r.lon,
        "min_msl_hPa": float(disk["msl"].min()) / 100,
        "max_wind_ms": float(ws.max()),
        "mean_t2m_K":  float(disk["t2m"].mean()),
    })
summary = pd.DataFrame(out)
print(len(out), "processed,", len(skipped), "skipped")

72 processed, 0 skipped


In [72]:
summary

,track_id,time,lat,lon,min_msl_hPa,max_wind_ms,mean_t2m_K
0,23,2004-02-17 00:00:00,63.75,-36.00,974.695000,26.002878,270.926331
1,23,2004-02-17 06:00:00,63.75,-33.25,973.248750,21.535357,272.102600
2,23,2004-02-17 12:00:00,64.50,-31.75,984.531875,21.304689,271.965179
3,23,2004-02-17 18:00:00,67.75,-27.25,988.453125,20.206255,271.644653
4,47,2004-02-17 00:00:00,69.00,24.25,1008.235000,8.481917,266.592407
...,...,...,...,...,...,...,...
67,667,2004-02-18 18:00:00,62.75,-37.00,1004.926875,9.176619,270.423096
68,667,2004-02-19 00:00:00,65.25,-38.50,1004.932500,12.907104,257.766174
69,667,2004-02-19 06:00:00,65.25,-38.25,1002.940000,14.479611,257.510925
70,667,2004-02-19 12:00:00,65.00,-36.50,995.646250,18.598255,267.579773


In [70]:
print("on_edge:", df.on_edge.sum())
print("outside file:", (~(df.lat.between(lat_lo, lat_hi) & df.lon.between(lon_lo, lon_hi))).sum())
print(df.assign(kept=df.index.isin(work.index)).groupby("track_id").kept.agg(["sum", "count"]))

on_edge: 30
outside file: 62
          sum  count
track_id            
23          4     11
47          1     16
65         11     11
75         11     11
113         9     12
167         0     13
347        16     16
605         0     10
641         8     10
667        12     12
673         0     12
